# 04 · FastAPI serving and streaming

Wrap an LLM in a production-shaped FastAPI service: typed request/response models, a streaming endpoint with
Server-Sent Events, upstream errors mapped to the right status codes, request ids, dependency injection so tests
can swap the model out, and a measurement of the sync-vs-async mistake that quietly caps a server at one request
at a time.

**Time:** ~30 min · **Runs:** offline on CPU in < 1 min · **Needs:** [01 · Async concurrency](01_async_concurrency_for_llm_calls.ipynb), [type hints and pydantic](../01_python_basics/13_type_hints_and_pydantic.ipynb)

## Why this matters in interviews

- "Build a chat API around an LLM" is one of the most common live-coding rounds, and FastAPI is the default answer.
  Interviewers look for pydantic models, async handlers, streaming, error mapping and a test.
- "Why async for an endpoint that calls an LLM?" has a measurable answer, and the classic trap (a blocking call
  inside `async def`) is easy to demonstrate: you will see it drop throughput 10x below.
- Streaming is where product latency is won ("feels slow" is usually time-to-first-token) and where the gotchas
  live: errors after the first byte, guardrails that need the whole answer, proxies that buffer.

## Interview questions this notebook answers

| ID | Question |
|----|----------|
| `tl08` | Why would you use async for an endpoint that calls an LLM? |
| `tl07` | Compare FastAPI and Streamlit. |
| `pd39` | What does "streaming" mean in a production LLM system, and what are the gotchas? |
| `in08` | What is streaming, and what does it actually improve? |
| `po33` | Accuracy is fine but users say the product feels slow. What is happening? |
| `tl20` | What does Pydantic do for an LLM application? |
| `po04` | Describe the layers of a production GenAI architecture and what each one owns. |
| `po17` | Why are timeouts more delicate in an LLM system than in a normal service? |
| `pd11` | What is async processing and when should an LLM feature be asynchronous? |

## Setup

In [ ]:
import sys
from pathlib import Path
for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / "llm_setup.py").exists():
        sys.path.insert(0, str(_p)); break
from llm_setup import client, MODEL, EMBED_MODEL, PROVIDER

In [ ]:
import asyncio
import json
import time
import uuid
from contextlib import asynccontextmanager
from types import SimpleNamespace

import httpx
import matplotlib.pyplot as plt
import numpy as np
import openai
from fastapi import Depends, FastAPI, HTTPException, Request
from fastapi.responses import JSONResponse, StreamingResponse
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field
from llm_setup import OFFLINE, make_async_client, make_client

client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": "hi"}], max_tokens=5)  # warm-up
import fastapi, pydantic
print(f"provider={PROVIDER}  fastapi {fastapi.__version__}  pydantic {pydantic.__version__}")

## 1. The shape of an LLM service

```
 client ──HTTP──>  FastAPI app                                                 ──>  LLM provider
                   ├─ middleware: request id, timing, (auth, rate limit)
                   ├─ pydantic: validate the request body (422 on bad input)
                   ├─ Depends(get_llm): the client, created once at startup
                   ├─ handler: async def, awaits the model
                   ├─ StreamingResponse (text/event-stream) for tokens as they arrive
                   └─ exception handlers: provider 429 -> 429, provider 5xx/timeout -> 502/504
```

**In plain English:** the web framework's job is the boring, critical part: reject bad input before spending
money, never let one slow model call block the others, turn provider errors into honest status codes, and give
every request an id you can find in the logs.

## 2. Request and response models

Pydantic models are the contract. They validate input (length limits are a cost control, not just hygiene),
document the API (FastAPI generates OpenAPI from them) and shape the output.

In [ ]:
class ChatRequest(BaseModel):
    message: str = Field(min_length=1, max_length=8_000, description="The user's message")
    max_tokens: int = Field(default=200, ge=1, le=1_000)
    temperature: float = Field(default=0.2, ge=0.0, le=2.0)

class Usage(BaseModel):
    prompt_tokens: int
    completion_tokens: int

class ChatResponse(BaseModel):
    reply: str
    model: str
    request_id: str
    finish_reason: str
    usage: Usage

print(ChatRequest(message="hi").model_dump())
try:
    ChatRequest(message="", max_tokens=5_000)
except pydantic.ValidationError as e:
    print(f"{e.error_count()} validation errors:", [err["loc"][0] for err in e.errors()])

## 3. The app

Five production details, each a few lines:

1. **Lifespan** creates one `AsyncOpenAI` client at startup and closes it at shutdown. Never create a client per
   request: you lose the connection pool, and a pool built on one event loop must not be reused on another.
2. **Dependency injection**: handlers ask for `llm = Depends(get_llm)`, so a test can override it.
3. **Request-id middleware**: reuse the caller's `x-request-id` or mint one, and return it on every response,
   including errors, so a user complaint becomes a log lookup.
4. **Exception handlers**: a provider 429 becomes our 429 (with `Retry-After`), provider timeouts 504, other
   provider failures 502. A 500 would claim *our* code crashed.
5. **Streaming**: open the upstream stream *before* returning `StreamingResponse`, so an upstream failure still gets
   a proper status code; after the first byte, errors can only be reported inside the stream.

In [ ]:
MAX_INPUT_TOKENS = 1_500

@asynccontextmanager
async def lifespan(app: FastAPI):
    app.state.llm = make_async_client(max_retries=1, timeout=30.0)   # one client for the process
    yield
    await app.state.llm.close()

app = FastAPI(title="chat-service", lifespan=lifespan)

@app.middleware("http")
async def request_id_and_timing(request: Request, call_next):
    request.state.request_id = request.headers.get("x-request-id") or uuid.uuid4().hex[:12]
    t0 = time.perf_counter()
    response = await call_next(request)
    response.headers["x-request-id"] = request.state.request_id
    response.headers["server-timing"] = f"app;dur={1000 * (time.perf_counter() - t0):.1f}"
    return response

def get_llm(request: Request):
    return request.app.state.llm

@app.exception_handler(openai.APIError)
async def upstream_error(request: Request, exc: openai.APIError):
    if isinstance(exc, openai.RateLimitError):
        status, headers = 429, {"retry-after": exc.response.headers.get("retry-after", "1")}
    elif isinstance(exc, openai.APITimeoutError):
        status, headers = 504, {}
    else:
        status, headers = 502, {}
    body = {"error": {"type": type(exc).__name__, "message": "The model provider failed; please retry."},
            "request_id": getattr(request.state, "request_id", None)}
    return JSONResponse(status_code=status, content=body, headers=headers)

def messages_for(req: ChatRequest):
    if len(req.message) / 4 > MAX_INPUT_TOKENS:          # rough token estimate; reject before paying for it
        raise HTTPException(status_code=413, detail=f"message too long (limit ~{MAX_INPUT_TOKENS} tokens)")
    return [{"role": "system", "content": "You are a concise assistant."}, {"role": "user", "content": req.message}]

In [ ]:
@app.get("/health")
async def health():
    return {"status": "ok", "model": MODEL}

@app.post("/chat", response_model=ChatResponse)
async def chat(req: ChatRequest, request: Request, llm=Depends(get_llm)):
    r = await llm.chat.completions.create(model=MODEL, messages=messages_for(req),
                                          max_tokens=req.max_tokens, temperature=req.temperature)
    choice = r.choices[0]
    return ChatResponse(reply=choice.message.content, model=r.model, request_id=request.state.request_id,
                        finish_reason=choice.finish_reason,
                        usage=Usage(prompt_tokens=r.usage.prompt_tokens, completion_tokens=r.usage.completion_tokens))

def sse(data: dict, event: str | None = None) -> str:
    """One Server-Sent Event: optional 'event:' line, a 'data:' line, then a blank line."""
    return (f"event: {event}\n" if event else "") + f"data: {json.dumps(data)}\n\n"

@app.post("/chat/stream")
async def chat_stream(req: ChatRequest, llm=Depends(get_llm)):
    stream = await llm.chat.completions.create(model=MODEL, messages=messages_for(req), max_tokens=req.max_tokens,
                                               temperature=req.temperature, stream=True)   # fails here -> real status code
    async def events():
        try:
            async for chunk in stream:
                if chunk.choices and chunk.choices[0].delta.content:
                    yield sse({"delta": chunk.choices[0].delta.content})
            yield "data: [DONE]\n\n"
        except openai.APIError as e:                      # after the first byte: report in-band
            yield sse({"type": type(e).__name__, "message": "stream interrupted"}, event="error")
    return StreamingResponse(events(), media_type="text/event-stream",
                             headers={"cache-control": "no-cache", "x-accel-buffering": "no"})

print([f"{sorted(r.methods)[0]} {r.path}" for r in app.routes if hasattr(r, "methods") and not r.path.startswith(("/docs", "/openapi", "/redoc"))])

## 4. Test it in-process with `TestClient`

`TestClient` runs the app in memory (no port, no server) and speaks HTTP to it. Used as a context manager it also
runs the lifespan, so the real client is created and closed.

In [ ]:
with TestClient(app) as tc:
    print("GET /health ->", tc.get("/health").json())
    r = tc.post("/chat", json={"message": "What is 17*23?"}, headers={"x-request-id": "demo-123"})
    print("POST /chat  ->", r.status_code, r.json())
    print("   headers   :", {k: r.headers[k] for k in ("x-request-id", "server-timing")})
    bad = tc.post("/chat", json={"message": "", "max_tokens": 5000})
    print("empty message ->", bad.status_code, [e["loc"][-1] for e in bad.json()["detail"]])
    huge = tc.post("/chat", json={"message": "word " * 1_500})
    print("7,500 chars   ->", huge.status_code, huge.json()["detail"])
assert r.status_code == 200 and r.json()["request_id"] == "demo-123"
assert bad.status_code == 422 and huge.status_code == 413

**Dependency override** is how you test the service without a model: swap `get_llm` for a fake with the same call
shape. The tests are free, deterministic and run in milliseconds, with any provider, and they can simulate the
failures a real provider will not produce on demand.

In [ ]:
REQ = httpx.Request("POST", "https://api.example.com/v1/chat/completions")

class FakeLLM:
    """Same call shape as AsyncOpenAI for chat.completions.create; canned output or a chosen failure."""
    def __init__(self, reply="Hello from the fake model.", fail=None, fail_after=None):
        self.reply, self.fail, self.fail_after, self.calls = reply, fail, fail_after, []
        self.chat = SimpleNamespace(completions=SimpleNamespace(create=self._create))
    async def _create(self, *, model, messages, stream=False, **kw):
        self.calls.append(messages)
        if self.fail:
            raise self.fail
        if stream:
            return self._stream()
        msg = SimpleNamespace(content=self.reply)
        return SimpleNamespace(model="fake-model", choices=[SimpleNamespace(message=msg, finish_reason="stop")],
                               usage=SimpleNamespace(prompt_tokens=7, completion_tokens=5))
    async def _stream(self):
        for i, word in enumerate(self.reply.split(" ")):
            if self.fail_after is not None and i == self.fail_after:
                raise openai.APIConnectionError(request=REQ)
            yield SimpleNamespace(choices=[SimpleNamespace(delta=SimpleNamespace(content=(" " if i else "") + word))])

def returns(obj):
    """A zero-argument dependency returning obj. (A lambda with a default argument would become a query parameter!)"""
    return lambda: obj

def status_error(cls, status, headers=None):
    return cls(f"HTTP {status}", response=httpx.Response(status, headers=headers or {}, request=REQ), body=None)

cases = {"healthy": FakeLLM(),
         "provider 429": FakeLLM(fail=status_error(openai.RateLimitError, 429, {"retry-after": "7"})),
         "provider 503": FakeLLM(fail=status_error(openai.InternalServerError, 503)),
         "provider timeout": FakeLLM(fail=openai.APITimeoutError(request=REQ))}
with TestClient(app) as tc:
    for name, fake in cases.items():
        app.dependency_overrides[get_llm] = returns(fake)
        r = tc.post("/chat", json={"message": "hi"})
        print(f"{name:<17} -> {r.status_code}  retry-after={r.headers.get('retry-after')}  body={r.json()}")
    app.dependency_overrides.clear()
    statuses = [tc.post("/chat", json={"message": "hi"}).status_code]
assert statuses == [200]

Offline we can also make the *real* client fail, end to end through the SDK (the `max_retries=1` in the lifespan
means two failures reach our handler):

In [ ]:
if OFFLINE:
    with TestClient(app) as tc:
        OFFLINE.fail_next(2, 429)
        r429 = tc.post("/chat", json={"message": "hi"})
        OFFLINE.fail_next(2, 500)
        r502 = tc.post("/chat", json={"message": "hi"})
    print("real SDK, provider 429 ->", r429.status_code, r429.json())
    print("real SDK, provider 500 ->", r502.status_code, r502.json())
    assert (r429.status_code, r502.status_code) == (429, 502)

## 5. Streaming with Server-Sent Events

SSE is plain HTTP: `Content-Type: text/event-stream`, then events separated by a blank line, each made of
`field: value` lines (`data:`, optionally `event:`, `id:`). It is one-directional (server to client), works through
most proxies, and browsers read it with `EventSource` or `fetch`. That is why OpenAI's own streaming API uses it.
Offline, the stand-in waits `OFFLINE.token_delay` between chunks, like a real model producing tokens.

In [ ]:
def parse_sse(text: str) -> list[dict]:
    events = []
    for block in text.strip().split("\n\n"):
        fields = dict(line.split(": ", 1) for line in block.splitlines() if ": " in line)
        events.append(fields)
    return events

if OFFLINE:
    OFFLINE.token_delay = 0.02
with TestClient(app) as tc:
    t0 = time.perf_counter()
    with tc.stream("POST", "/chat/stream", json={"message": "Summarise this.\n\nStreaming sends tokens as they are "
                                                 "generated. It changes perceived latency. Total time is unchanged."}) as resp:
        body = "".join(resp.iter_text())
    elapsed = time.perf_counter() - t0
events = parse_sse(body)
print("content-type:", resp.headers["content-type"])
print("first events:", events[:3])
print("last event  :", events[-1])
text = "".join(json.loads(e["data"])["delta"] for e in events if e.get("data", "").startswith("{"))
print(f"reassembled : {text!r}  ({len(events) - 1} deltas in {elapsed:.2f} s)")
assert resp.headers["content-type"].startswith("text/event-stream") and events[-1]["data"] == "[DONE]"

`TestClient` buffers the body, so to *see* what streaming buys, time the model client directly. A model generates
tokens one after another either way; a non-streaming call simply returns when the **last** token is done, a streaming
call hands you each token as it is produced. One streamed call shows both. (Offline: 0.3 s before the first token,
30 ms per token after it. The stand-in skips generation time for non-streamed calls, which a real model cannot.)

In [ ]:
if OFFLINE:
    OFFLINE.latency, OFFLINE.token_delay = 0.3, 0.03
probe = make_async_client()
long_q = [{"role": "user", "content": "Summarise this.\n\n" + "The service streams tokens to the browser as they arrive. "
           "Users start reading while the model is still writing. The total time does not change at all."}]
t0, arrivals = time.perf_counter(), []
async for chunk in await probe.chat.completions.create(model=MODEL, messages=long_q, max_tokens=60, stream=True):
    if chunk.choices and chunk.choices[0].delta.content:
        arrivals.append(time.perf_counter() - t0)
t_block = arrivals[-1]                     # when a non-streaming response could be returned
print(f"streaming    : first text after {arrivals[0]:.2f} s ({len(arrivals)} tokens, last at {arrivals[-1]:.2f} s)")
print(f"non-streaming: first text after ~{t_block:.2f} s (the whole answer at once)")
assert arrivals[0] < t_block / 2

fig, ax = plt.subplots(figsize=(9, 2.8))
ax.barh(1, t_block, color="#adb5bd"); ax.text(t_block + 0.02, 1, "whole answer appears", va="center", fontsize=9)
ax.scatter(arrivals, np.zeros(len(arrivals)), marker="|", s=300, color="#2f9e44")
ax.text(arrivals[0], 0.35, f"first token {arrivals[0]:.2f} s", fontsize=9, color="#2f9e44")
ax.set_yticks([0, 1], ["streaming\n(each tick = a token)", "non-streaming"]); ax.set_ylim(-0.6, 1.6)
ax.set(xlabel="seconds since the request", title="Streaming: same total time, text appears much sooner")
plt.show()

Two failure modes, and the difference between them is the design point (`pd39`):

In [ ]:
with TestClient(app) as tc:
    app.dependency_overrides[get_llm] = lambda: FakeLLM(fail=status_error(openai.RateLimitError, 429))
    before_first_byte = tc.post("/chat/stream", json={"message": "hi"})
    app.dependency_overrides[get_llm] = lambda: FakeLLM(reply="one two three four five", fail_after=3)
    mid_stream = tc.post("/chat/stream", json={"message": "hi"})
    app.dependency_overrides.clear()
print("fails before the first byte ->", before_first_byte.status_code, before_first_byte.json()["error"]["type"])
print("fails mid-stream            ->", mid_stream.status_code, parse_sse(mid_stream.text))
assert before_first_byte.status_code == 429
assert mid_stream.status_code == 200 and parse_sse(mid_stream.text)[-1].get("event") == "error"

Once the first byte is sent, the status code is already 200, so the client must understand an in-band `error`
event and show a partial answer honestly. Other streaming gotchas worth naming:

| Gotcha | What happens | Fix |
|---|---|---|
| Output guardrails | you cannot un-show tokens that failed a check | buffer and release by sentence, or run the check on chunks, or accept the risk explicitly |
| Proxies and load balancers | nginx buffers responses; idle timeouts kill long streams | `X-Accel-Buffering: no`, raise idle timeouts, send keep-alive comments |
| Structured output | JSON is not valid until the end | stream prose, or parse incrementally, or do not stream |
| Timeouts | one total timeout is wrong for a stream | separate time-to-first-token, inter-token stall and total limits (`po17`) |
| Client disconnects | you keep paying for tokens nobody reads | check `await request.is_disconnected()` and stop the upstream stream |

Streaming does not make the answer faster: total time and cost are the same, and the first token takes as long
as ever. It changes **perceived** latency: text starts appearing after the TTFT instead of after the whole answer
(`in08`, `po33`).

## 6. The sync-vs-async trap, measured

Four ways to write the same handler. We fire 10 concurrent requests at each through `httpx.ASGITransport`, which
runs the app on this notebook's own event loop, and measure throughput. Offline each model call takes 0.3 s.

In [ ]:
if OFFLINE:
    OFFLINE.latency, OFFLINE.token_delay = 0.3, 0.0
sync_llm, async_llm = make_client(), make_async_client()
PROMPT = [{"role": "user", "content": "What is 6*7?"}]

def call_sync():
    return sync_llm.chat.completions.create(model=MODEL, messages=PROMPT, max_tokens=10).choices[0].message.content

bench = FastAPI()

@bench.get("/sync-def")                   # sync handler: FastAPI runs it in a thread pool
def sync_def():
    return {"r": call_sync()}

@bench.get("/async-blocking")             # BAD: a blocking SDK call inside async def
async def async_blocking():
    return {"r": call_sync()}

@bench.get("/async-client")               # async handler + async client
async def async_client():
    r = await async_llm.chat.completions.create(model=MODEL, messages=PROMPT, max_tokens=10)
    return {"r": r.choices[0].message.content}

@bench.get("/async-to-thread")            # stuck with a sync library? push it to a thread
async def async_to_thread():
    return {"r": await asyncio.to_thread(call_sync)}

In [ ]:
N_CONC = 10 if OFFLINE else 4              # fewer paid calls with a real provider

async def load_test(path, n=N_CONC):
    async with httpx.AsyncClient(transport=httpx.ASGITransport(app=bench), base_url="http://bench") as hc:
        t0 = time.perf_counter()
        async def one():
            r = await hc.get(path)
            assert r.status_code == 200
            return time.perf_counter() - t0
        finish = await asyncio.gather(*(one() for _ in range(n)))
    return time.perf_counter() - t0, sorted(finish)

shapes = ["/sync-def", "/async-blocking", "/async-client", "/async-to-thread"]
results = {p: await load_test(p) for p in shapes}
for p, (t, _) in results.items():
    print(f"{p:<17} {t:5.2f} s for {N_CONC} requests -> {N_CONC / t:5.1f} req/s")
rps = {p: N_CONC / t for p, (t, _) in results.items()}
assert rps["/async-blocking"] < rps["/async-client"] / 2.5, "a blocking call inside async def serialises the server"
assert min(rps["/sync-def"], rps["/async-to-thread"]) > rps["/async-blocking"] * 2.5

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
colors = ["#74c0fc", "#e03131", "#2f9e44", "#2f9e44"]
labels = ["def +\nsync client", "async def +\nblocking call (BAD)", "async def +\nasync client", "async def +\nto_thread"]
bars = axes[0].bar(labels, list(rps.values()), color=colors)
axes[0].bar_label(bars, fmt="%.1f")
axes[0].set(ylabel="requests per second", title=f"{N_CONC} concurrent requests, one process")
for p, c, lab in [("/async-blocking", "#e03131", "async def + blocking call"), ("/async-client", "#2f9e44", "async def + async client")]:
    axes[1].step(results[p][1], np.arange(1, N_CONC + 1), where="post", color=c, lw=2, label=lab)
axes[1].set(xlabel="seconds since the burst", ylabel="requests completed", title="Blocking: served one at a time")
axes[1].legend(); plt.tight_layout(); plt.show()

What happened:

| Handler | What FastAPI does | Result |
|---|---|---|
| `def` + sync client | runs it in a thread pool (AnyIO's default limit is 40 threads) | concurrent, until the pool is full |
| `async def` + blocking call | runs it **on the event loop**; the blocking call freezes the loop | one request at a time: `N x latency` |
| `async def` + async client | awaits yield the loop; hundreds can be in flight | concurrent, limited by the provider |
| `async def` + `asyncio.to_thread` | the blocking call moves to a worker thread | concurrent, up to the executor's thread count |

So "just make it async" is not the rule; "never block inside `async def`" is. A plain `def` handler is a safe
choice with a sync SDK; `async def` + an async client scales furthest (`tl08`).

## 7. Running it for real

The notebook never opens a port. In a real deployment you save the app as `app.py` and run it with an ASGI server:

```bash
pip install fastapi uvicorn openai
uvicorn app:app --host 0.0.0.0 --port 8000 --workers 4        # 4 processes, each with its own event loop
# or behind gunicorn's process manager:
gunicorn app:app -k uvicorn.workers.UvicornWorker --workers 4 --timeout 120
curl -N -X POST localhost:8000/chat/stream -H 'content-type: application/json' -d '{"message": "hi"}'
```

- **Workers** are processes: each has its own event loop, client and connection pool. For I/O-bound LLM work, a few
  async workers per CPU is plenty; the real limit is the provider quota, so cap concurrency per worker (a semaphore)
  and share rate-limit state across workers (Redis).
- **Timeouts at every layer**: client, load balancer, server and SDK. The stingiest one kills long generations.
- **Long jobs** (a 40 s agent run, a batch of documents) should not hold a request open: return a job id and use a
  queue (`pd11`); `BackgroundTasks` is fine for small fire-and-forget work only.
- Interactive docs are generated at `/docs` from the pydantic models.
- **Streamlit** is for internal tools and demos (it reruns the whole script per interaction, has no routing or real
  auth); the production shape is a Streamlit or React front end calling a FastAPI service (`tl07`).

## 8. The complete `app.py`

Everything above in one file you can run with `uvicorn app:app` (it uses the plain OpenAI SDK; point `base_url`
at Ollama, vLLM or a LiteLLM proxy to change providers):

```python
# app.py  -  run: uvicorn app:app --port 8000
import json, os, time, uuid
from contextlib import asynccontextmanager

import openai
from fastapi import Depends, FastAPI, HTTPException, Request
from fastapi.responses import JSONResponse, StreamingResponse
from openai import AsyncOpenAI
from pydantic import BaseModel, Field

MODEL = os.getenv("MODEL", "gpt-4.1-mini")
MAX_INPUT_TOKENS = 1_500


class ChatRequest(BaseModel):
    message: str = Field(min_length=1, max_length=8_000)
    max_tokens: int = Field(default=200, ge=1, le=1_000)
    temperature: float = Field(default=0.2, ge=0.0, le=2.0)


class Usage(BaseModel):
    prompt_tokens: int
    completion_tokens: int


class ChatResponse(BaseModel):
    reply: str
    model: str
    request_id: str
    finish_reason: str
    usage: Usage


@asynccontextmanager
async def lifespan(app: FastAPI):
    app.state.llm = AsyncOpenAI(max_retries=1, timeout=30.0)   # reads OPENAI_API_KEY / OPENAI_BASE_URL
    yield
    await app.state.llm.close()


app = FastAPI(title="chat-service", lifespan=lifespan)


@app.middleware("http")
async def request_id_and_timing(request: Request, call_next):
    request.state.request_id = request.headers.get("x-request-id") or uuid.uuid4().hex[:12]
    t0 = time.perf_counter()
    response = await call_next(request)
    response.headers["x-request-id"] = request.state.request_id
    response.headers["server-timing"] = f"app;dur={1000 * (time.perf_counter() - t0):.1f}"
    return response


def get_llm(request: Request) -> AsyncOpenAI:
    return request.app.state.llm


@app.exception_handler(openai.APIError)
async def upstream_error(request: Request, exc: openai.APIError):
    if isinstance(exc, openai.RateLimitError):
        status, headers = 429, {"retry-after": exc.response.headers.get("retry-after", "1")}
    elif isinstance(exc, openai.APITimeoutError):
        status, headers = 504, {}
    else:
        status, headers = 502, {}
    body = {"error": {"type": type(exc).__name__, "message": "The model provider failed; please retry."},
            "request_id": getattr(request.state, "request_id", None)}
    return JSONResponse(status_code=status, content=body, headers=headers)


def messages_for(req: ChatRequest):
    if len(req.message) / 4 > MAX_INPUT_TOKENS:
        raise HTTPException(status_code=413, detail=f"message too long (limit ~{MAX_INPUT_TOKENS} tokens)")
    return [{"role": "system", "content": "You are a concise assistant."}, {"role": "user", "content": req.message}]


@app.get("/health")
async def health():
    return {"status": "ok", "model": MODEL}


@app.post("/chat", response_model=ChatResponse)
async def chat(req: ChatRequest, request: Request, llm: AsyncOpenAI = Depends(get_llm)):
    r = await llm.chat.completions.create(model=MODEL, messages=messages_for(req),
                                          max_tokens=req.max_tokens, temperature=req.temperature)
    choice = r.choices[0]
    return ChatResponse(reply=choice.message.content, model=r.model, request_id=request.state.request_id,
                        finish_reason=choice.finish_reason,
                        usage=Usage(prompt_tokens=r.usage.prompt_tokens, completion_tokens=r.usage.completion_tokens))


def sse(data: dict, event: str | None = None) -> str:
    return (f"event: {event}\n" if event else "") + f"data: {json.dumps(data)}\n\n"


@app.post("/chat/stream")
async def chat_stream(req: ChatRequest, request: Request, llm: AsyncOpenAI = Depends(get_llm)):
    stream = await llm.chat.completions.create(model=MODEL, messages=messages_for(req), max_tokens=req.max_tokens,
                                               temperature=req.temperature, stream=True)

    async def events():
        try:
            async for chunk in stream:
                if await request.is_disconnected():          # stop paying for tokens nobody reads
                    await stream.close()
                    return
                if chunk.choices and chunk.choices[0].delta.content:
                    yield sse({"delta": chunk.choices[0].delta.content})
            yield "data: [DONE]\n\n"
        except openai.APIError as e:
            yield sse({"type": type(e).__name__, "message": "stream interrupted"}, event="error")

    return StreamingResponse(events(), media_type="text/event-stream",
                             headers={"cache-control": "no-cache", "x-accel-buffering": "no"})
```

## Try it yourself

**1.** Find the thread-pool ceiling: send 120 concurrent requests to `/sync-def` and `/async-client`. Predict first:
AnyIO's pool runs at most 40 sync handlers at once, so the sync version needs three waves of 0.3 s.

In [ ]:
# Solution — try it yourself first, then run this
if OFFLINE:
    t_sync, _ = await load_test("/sync-def", n=120)
    t_async, _ = await load_test("/async-client", n=120)
    print(f"120 requests: def + sync client {t_sync:.2f} s (3 waves of 40 -> at least 0.9 s), async client {t_async:.2f} s")
    assert t_sync >= 0.85 and t_async < t_sync
else:
    print("Skipped with a real provider (it would make 240 paid calls).")

**2.** Add API-key auth as a dependency: a request without the right `x-api-key` header must get 401 before the
model is called. Test it with the fake LLM and check the fake was never called.

In [ ]:
# Solution — try it yourself first, then run this
API_KEYS = {"secret-key-1": "tenant-acme"}

def require_tenant(request: Request) -> str:
    tenant = API_KEYS.get(request.headers.get("x-api-key", ""))
    if tenant is None:
        raise HTTPException(status_code=401, detail="missing or invalid API key")
    return tenant

@app.post("/v2/chat")
async def chat_v2(req: ChatRequest, tenant: str = Depends(require_tenant), llm=Depends(get_llm)):
    r = await llm.chat.completions.create(model=MODEL, messages=messages_for(req))
    return {"tenant": tenant, "reply": r.choices[0].message.content}

fake = FakeLLM()
with TestClient(app) as tc:
    app.dependency_overrides[get_llm] = lambda: fake
    no_key = tc.post("/v2/chat", json={"message": "hi"})
    with_key = tc.post("/v2/chat", json={"message": "hi"}, headers={"x-api-key": "secret-key-1"})
    app.dependency_overrides.clear()
print(no_key.status_code, no_key.json(), "|", with_key.status_code, with_key.json(), "| model calls:", len(fake.calls))
assert no_key.status_code == 401 and with_key.status_code == 200 and len(fake.calls) == 1

**3.** Write the test a reviewer would ask for: the streaming endpoint's deltas, joined, must equal the fake
model's reply, and the stream must end with `[DONE]`.

In [ ]:
# Solution — try it yourself first, then run this
fake = FakeLLM(reply="streams are just text")
with TestClient(app) as tc:
    app.dependency_overrides[get_llm] = lambda: fake
    ev = parse_sse(tc.post("/chat/stream", json={"message": "hi"}).text)
    app.dependency_overrides.clear()
joined = "".join(json.loads(e["data"])["delta"] for e in ev[:-1])
print(repr(joined), ev[-1])
assert joined == fake.reply and ev[-1]["data"] == "[DONE]"

if OFFLINE:                               # reset the stand-in's knobs
    OFFLINE.latency, OFFLINE.token_delay = 0.0, 0.0

## Say it in an interview

- **30 seconds:** "A FastAPI service with pydantic request and response models, one async OpenAI client created in
  the lifespan and injected with `Depends`, a request-id middleware, and exception handlers that turn provider 429s
  into 429 with `Retry-After` and provider failures into 502/504. A `/chat/stream` endpoint returns
  `text/event-stream`; I open the upstream stream before returning so early failures get real status codes, and
  report later failures as an in-band error event. Tests override the LLM dependency with a fake."
- **Why async:** the handler spends ~99% of its time waiting on the provider, so one async worker holds hundreds of
  requests. In this notebook a blocking call inside `async def` dropped throughput from ~25 to ~3 req/s.
- **Streaming:** improves perceived latency (time until text appears), not total time or cost, and not TTFT itself.
- **Follow-ups:** workers vs concurrency, timeouts per layer and deadline propagation, long jobs on a queue with a
  job id, auth and per-tenant rate limits as dependencies, disconnect handling, where guardrails sit.
- **Traps:** a sync SDK inside `async def`; a client per request; returning 500 for provider errors; one total
  timeout for a stream; exact-string assertions on model output in tests.

## Next

- [05 · Latency percentiles and cost](05_latency_percentiles_p50_p95_and_cost.ipynb): TTFT, TPOT and what to put in an SLO.
- [08 · Observability](08_observability_logging_and_tracing.ipynb): structured logs and traces keyed by the request id.
- [Chat API with FastAPI (interview round)](../17_interview_coding_rounds/03_chat_api_with_fastapi.ipynb): the 45-minute version.
- Related: [02 · Retries and circuit breakers](02_retries_backoff_rate_limits_circuit_breakers.ipynb) · [07 · Guardrails](07_guardrails_pii_and_prompt_injection.ipynb).
- Theory: [AI system design concepts](../../ai_system_design_concepts/index.html) · [interview bank](../../ai_interview_prep/index.html).